# ML-10 — Content Action Playbook

This notebook turns our validated model output into a content action playbook. We load our prioritized refresh queue, map observations to reason codes, define human review checks, detail the cost/value trade-offs, and export artifacts for the research paper.

## 1. Ranked actions + reason codes

### The Decay/Refresh Insight
Organic search visibility decay is non-linear and context-dependent. A static rule like "refresh every page older than 180 days" leads to wasted editorial resources on low-volume pages while missing high-opportunity pages undergoing rapid rank erosion. Visibility (impressions) and average position interact heavily with freshness: a high-impression page experiencing a slide in position (e.g., from rank 3 to rank 8) represents a high-priority opportunity because a small relevance update can prevent major traffic loss.

### Archetype &rarr; Action Mapping
We map page performance profiles to targeted editorial interventions:
1. **Stale & High Visibility (Archetype: `stale_visible_page`):** Impressions &ge; 500, days since update &ge; 180 &rarr; Action: `refresh`
2. **Visible & Low CTR (Archetype: `low_ctr_visible_page`):** Impressions &ge; 500, position 1-20, CTR < 0.5% &rarr; Action: `refresh_and_review_ctr` (Optimize title, meta description, or rich snippets)
3. **Thin & High Visibility (Archetype: `thin_visible_page`):** Impressions &ge; 250, word count < 1200 &rarr; Action: `expand_and_refresh` (Add depth, answer user questions, cover missing subtopics)
4. **Low User Engagement (Archetype: `low_engagement_visible_page`):** Sessions &ge; 30, engagement or scroll rate < 30% &rarr; Action: `refresh_and_review_engagement` (Improve layout, internal links, or readability)

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# Locate repository root
repo_root = next(
    (candidate for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent] if (candidate / "scripts" / "ml_utils.py").exists()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Could not locate the repository root.")

queue_path = repo_root / "outputs" / "refresh_queue.csv"
if not queue_path.exists():
    queue_path = repo_root / "outputs" / "refresh_queue_sample.csv"

queue = pd.read_csv(queue_path)
print(f"Loaded queue: {len(queue):,} rows")
display(queue[["final_rank", "content_id", "client_id", "final_refresh_score", "suggested_action", "final_reason_codes", "confidence"]].head(10))

Loaded queue: 30,000 rows


,final_rank,content_id,client_id,final_refresh_score,suggested_action,final_reason_codes,confidence
0,1,content_1f080331fa2b,client_3fdba35f04,81.734212,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,high
1,2,content_d6570c51c9bd,client_3fdba35f04,81.603243,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,medium
2,3,content_6aa43079fb0c,client_3fdba35f04,81.544618,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,high
3,4,content_72e800a9c214,client_3fdba35f04,81.169731,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,high
4,5,content_e04eb9549989,client_3fdba35f04,80.957565,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,medium
5,6,content_b69288c5e701,client_3fdba35f04,80.798090,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,high
6,7,content_9b6df29f7889,client_3fdba35f04,80.650656,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,high
7,8,content_ba6f9dfcbca1,client_3fdba35f04,80.432641,refresh,declining_with_demand|model_decline_risk|visib...,medium
8,9,content_4d76cdb3387b,client_3fdba35f04,80.428403,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,medium
9,10,content_b4f35d640b1c,client_3fdba35f04,80.428243,refresh,declining_with_demand|model_decline_risk|visib...,medium


## 2. Intended use and limits

This queue acts as a decision-support system for editorial teams, not as an automatic publishing engine. It identifies existing high-visibility pages that show traffic/rank decline or severe staleness.

**Limits:** The model cannot evaluate brand new pages or predict success for unwritten content. It operates under the assumption that past demand is a proxy for future potential, which may fail during sudden topic shifts or search engine design updates.

In [2]:
print("Distribution of suggested actions:")
print(queue["suggested_action"].value_counts(dropna=False))
print("\nDistribution of confidence bands:")
print(queue["confidence"].value_counts(dropna=False))

Distribution of suggested actions:
suggested_action
monitor                          13083
refresh                           8188
refresh_and_review_ctr            6654
refresh_and_review_engagement     1993
expand_and_refresh                  82
Name: count, dtype: int64

Distribution of confidence bands:
confidence
low       15000
medium    11398
high       3602
Name: count, dtype: int64


## 3. Human review + the no-go list

### Cost/Value Thinking
- **False Positive Cost:** Low. Represents 1-2 hours of writer/editor time spent review/updating a page that was stable. However, there is a minor risk of rank volatility if unnecessary changes disrupt existing rankings.
- **False Negative Cost:** High. Represents unmitigated traffic and conversion declines, eroding organic search revenue.
- **Resource Optimization:** Blended scoring allows the editorial team to focus only on pages with high baseline demand (impressions &ge; 500) where updates yield the highest ROI.

### Human Checklist:
1. **Recent updates:** Verify if the page has been updated since the last crawl/scrape snapshot in the dataset.
2. **Keyword Intent:** Verify if the keyword target is still relevant.
3. **Competitor Check:** Check if competing articles are high-quality, requiring a structural rewrite rather than a simple refresh.

### No-Go List (Do NOT Automate):
- Do **NOT** automatically update pages without editorial oversight.
- Do **NOT** publish AI-generated text blindly.
- Do **NOT** bulk-update low-confidence recommendations without individual inspection.

In [3]:
print("Low-confidence examples requiring careful manual review:")
low_conf = queue[queue["confidence"] == "low"].head(5)
display(low_conf[["content_id", "final_refresh_score", "suggested_action", "final_reason_codes", "impressions_90d"]])

Low-confidence examples requiring careful manual review:


,content_id,final_refresh_score,suggested_action,final_reason_codes,impressions_90d
15000,content_7f305454dcb7,53.606232,refresh,declining_with_demand|model_decline_risk,157
15001,content_75e9c39688ca,53.604027,refresh,declining_with_demand,190
15002,content_c6c5c6c43e94,53.603301,refresh_and_review_ctr,declining_with_demand|page_one_decay_risk|low_...,1936
15003,content_20d68ef89496,53.603067,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|ctr...,2440
15004,content_be3bdd2f8fa2,53.601736,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|vis...,726


## 4. Monitoring / retrain triggers

We track post-refresh performance over 30 and 90 days. 

### Retrain Triggers:
- **Performance Degradation:** Retrain if the out-of-sample precision@50 drops by more than 10% from validation levels.
- **Concept Drift:** Retrain if global click-through patterns shift following major search engine algorithm changes.
- **Cadence:** Retrain quarterly to capture seasonality.

In [4]:
base_rate = queue["is_declining_label"].mean() if "is_declining_label" in queue.columns else np.nan
print(f"Baseline decline rate: {base_rate:.3f}")
print(f"Average CTR across scored queue: {queue['ctr'].mean():.3f}%")
print(f"Average search position: {queue['avg_position'].mean():.2f}")

Baseline decline rate: 0.542
Average CTR across scored queue: 0.511%
Average search position: 16.34


## 5. Exports for the paper

I export a sample queue and charts to `work/outputs/` and `work/figures/` for use in the final capstone report and static research page.

In [5]:
import sys
sys.path.insert(0, str(repo_root / "scripts"))
from ml_utils import write_json, simple_svg_bar_chart

work_outputs = repo_root / "work" / "outputs"
work_outputs.mkdir(parents=True, exist_ok=True)
work_figures = repo_root / "work" / "figures"
work_figures.mkdir(parents=True, exist_ok=True)

# Export ranked queue sample
queue.head(100).to_csv(work_outputs / "ranked_queue_sample.csv", index=False)
print(f"Exported queue sample to: {work_outputs / 'ranked_queue_sample.csv'}")

# Export metadata JSON
metadata = {
    "total_scored": int(len(queue)),
    "high_confidence_count": int((queue["confidence"] == "high").sum()),
    "action_counts": queue["suggested_action"].value_counts().to_dict(),
    "confidence_counts": queue["confidence"].value_counts().to_dict(),
}
write_json(work_outputs / "playbook_metadata.json", metadata)
print(f"Exported metadata to: {work_outputs / 'playbook_metadata.json'}")

# Export action distribution chart
action_counts = queue["suggested_action"].value_counts()
simple_svg_bar_chart(
    "Action Distribution in Scored Queue",
    action_counts.index.tolist(),
    [float(val) for val in action_counts.values],
    work_figures / "action_mix.svg",
    color="#426B69"
)
print(f"Exported chart to: {work_figures / 'action_mix.svg'}")

Exported queue sample to: c:\Users\kunal\Documents\Project\Internship\FlyRank\Google-Search-Ranking-Discoverability\work\outputs\ranked_queue_sample.csv
Exported metadata to: c:\Users\kunal\Documents\Project\Internship\FlyRank\Google-Search-Ranking-Discoverability\work\outputs\playbook_metadata.json
Exported chart to: c:\Users\kunal\Documents\Project\Internship\FlyRank\Google-Search-Ranking-Discoverability\work\figures\action_mix.svg


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.